# Finished Product — Data Cleaning & Normalization

Analyze and normalize the `database-finished-product-sic-jac.csv` reference data from SIC JAC.
Focus on categories, sub-categories, product type, concentration and code patterns.

**Source:** `docs/reference/warehouse/raw/database-finished-product-sic-jac.csv`

In [ ]:
from pathlib import Path
import re
import pandas as pd

source = Path(r"C:\Users\Alexander\Desktop\illampu\yarn-epr\docs\reference\warehouse\raw\database-finished-product-sic-jac.csv")
raw_text = source.read_text(encoding="utf-8", errors="replace")
print(f"Loaded {source} ({len(raw_text.splitlines())} lines)")
print("\n=== Preview ===")
for i, line in enumerate(raw_text.splitlines()[:40], start=1):
    print(f"{i:02d}: {line}")


def parse_section(text, header):
    lines = text.splitlines()
    result = []
    collecting = False
    for line in lines:
        if collecting:
            if not line.strip():
                break
            if line.strip().upper().startswith("SUB CATEGORIAS") and header.upper() in line.upper():
                continue
            result.append(line.rstrip())
        elif header.upper() in line.upper():
            collecting = True
    return result

finished_product_subcats = parse_section(raw_text, "SUB CATEGORIAS DE ALMACEN PRODUCTO TERMINADO:")
returns_subcats = parse_section(raw_text, "SUB CATEGORIAS DE DEVOLUCIONES DE LOTES:")

rows = []
for section, label in [("finished", line) for line in finished_product_subcats] + [("returns", line) for line in returns_subcats]:
    match = re.match(r'^\s*\d+\.\s*(.+)$', label)
    value = match.group(1).strip() if match else label.strip()
    rows.append({"section": section, "raw": value})

df = pd.DataFrame(rows)
print('\nLoaded rows:')
print(df.to_string(index=False))

Loaded C:\Users\Alexander\Desktop\illampu\yarn-epr\docs\reference\warehouse\raw\database-finished-product-sic-jac.csv (65 lines)

=== Preview ===
01: CATEGORIAS:
02: 1. Almacen producto terminado 
03: 2. Devoluciones de lotes
04: SUB CATEGORIAS DE ALMACEN PRODUCTO TERMINADO:
05: 1. TITULO 2/18
06: 2. TITULO 2/32
07: 3. TITULO 2/30
08: 4. RECUPERADO
09: 5. TIPO N
10: 6. ALPACA
11: 7. MIXTO
12: 8. TIPO CH
13: 9. MADEJAS OBSERVADAS
14: 10. TITULO 2/9                                                                                                                                          
15: 11. OVILLOS 2/9
16: 12. OVILLOS 4/9
17: 13. OVILLOS 3/5
18: 14. OVILLOS 3/11
19: 15. OVILLOS 3/20
20: 16. OVILLOS TIPO N 2/8
21: 17. MADEJITAS 3/15
22: 18. TITULO 3/15
23: 19. OVILLOS 3/15
24: 20. MADEJITAS MIXTAS
25: 21. TITULO 2/9
26: 22. TITULO 3/11 CH
27: 23. titulo 2/9 CH
28: 24. TITULO 4/9 CH
29: 25. TITULO 3/11
30: 26. TITULO 3/15 CH
31: 27. TITULO 4/9
32: 28. TITULO 1/36
33: 29. TITULO 2/13 TIPO

## 1. Normalizar las sub-categorías

Dividir cada etiqueta en `tipo`, `concentracion` y `extra` para generar filtros por `TITULO`, `OVILLOS`, `RECUPERADO`, `MIXTO`, `ALPACA`, etc.

In [ ]:
pattern = re.compile(
    r'(?P<tipo>^[A-ZÁÉÍÓÚÑ]+(?:\s+[A-ZÁÉÍÓÚÑ]+)?)(?:\s+)?(?P<concentracion>\d+/\d+)?(?:\s+(?P<extra>.+))?$',
    re.IGNORECASE,
)


def split_label(value):
    if not isinstance(value, str):
        return None, None, None
    value = value.strip().upper()
    if value in {"RECUPERADO", "MIXTO", "ALPACA", "STOLL"}:
        return value, None, None
    match = pattern.match(value)
    if not match:
        return value, None, None
    tipo = match.group("tipo").strip() if match.group("tipo") else None
    concentracion = match.group("concentracion")
    extra = match.group("extra").strip() if match.group("extra") else None
    return tipo, concentracion, extra

parsed = df["raw"].apply(lambda x: pd.Series(split_label(x), index=["tipo", "concentracion", "extra"]))
df = pd.concat([df, parsed], axis=1)
print(df.to_string(index=False))

 section                                      raw               tipo concentracion                     extra
finished                              TITULO 2/18             TITULO          2/18                       NaN
finished                              TITULO 2/32             TITULO          2/32                       NaN
finished                              TITULO 2/30             TITULO          2/30                       NaN
finished                               RECUPERADO         RECUPERADO           NaN                       NaN
finished                                   TIPO N             TIPO N           NaN                       NaN
finished                                   ALPACA             ALPACA           NaN                       NaN
finished                                    MIXTO              MIXTO           NaN                       NaN
finished                                  TIPO CH            TIPO CH           NaN                       NaN
finished           

In [ ]:
print('\nTipo counts:')
print(df['tipo'].value_counts(dropna=False))
print('\nConcentracion counts:')
print(df['concentracion'].value_counts(dropna=False))
print('\nExtra qualifiers:')
print(df['extra'].value_counts(dropna=False))


Tipo counts:
tipo
TITULO                47
OVILLOS                6
STOLL                  3
OVILLOS TIPO           2
MADEJITAS              2
TITULO CH              2
2/13 TIPO N            2
RECUPERADO             1
TIPO N                 1
ALPACA                 1
MIXTO                  1
TIPO CH                1
MADEJAS OBSERVADAS     1
MADEJITAS MIXTAS       1
TITULO FANTASIA        1
2/25                   1
SUB CATEGORIAS         1
Name: count, dtype: int64

Concentracion counts:
concentracion
NaN     19
3/15    10
4/9      5
3/11     5
2/9      4
2/17     4
2/18     3
2/32     3
2/12     3
2/11     3
2/10     3
2/30     2
3/5      1
3/20     1
1/36     1
2/13     1
1/32     1
2/15     1
2/20     1
2/28     1
2/19     1
1/20     1
Name: count, dtype: int64

Extra qualifiers:
extra
NaN                          60
CH                            6
TIPO N                        2
MADEJITAS                     2
N 2/8                         1
N 2/13                        1
SN      

In [ ]:
# Filtros de ejemplo
df_titulo = df[df['tipo'] == 'TITULO']
df_ovillos = df[df['tipo'] == 'OVILLOS']
df_recuperado = df[df['tipo'] == 'RECUPERADO']
df_2_18 = df[df['concentracion'] == '2/18']

print('TITULO rows:', len(df_titulo))
print('OVILLOS rows:', len(df_ovillos))
print('RECUPERADO rows:', len(df_recuperado))
print('2/18 rows:', len(df_2_18))

print('\nEjemplo TITULO 2/18:')
print(df[(df['tipo'] == 'TITULO') & (df['concentracion'] == '2/18')].to_string(index=False))

TITULO rows: 47
OVILLOS rows: 6
RECUPERADO rows: 1
2/18 rows: 3

Ejemplo TITULO 2/18:
 section         raw   tipo concentracion extra
finished TITULO 2/18 TITULO          2/18   NaN
finished TITULO 2/18 TITULO          2/18   NaN
 returns TITULO 2/18 TITULO          2/18   NaN


## 2. Siguientes pasos

- Ajustar el patrón si aparecen más variaciones en las categorías.
- Agregar exportación a CSV para resultados filtrados.
- Usar `df` normalizado para informes de inventario por tipo y concentración.